In [1]:
# FIX: Downgrade numpy to resolve binary incompatibility with pandas and langchain
!pip install -q "numpy<2.0.0"

import os
import pandas as pd
import torch
import csv
from typing import List, Dict, Any
from google.colab import userdata
from transformers import AutoTokenizer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

CSV_FILE_PATH = "/content/medquad.csv"

class MedQuadAssistant:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

        print("Initializing Gemini 2.5 Flash...")
        self.llm = ChatGoogleGenerativeAI(
            model="models/gemini-2.5-flash",
            google_api_key=userdata.get('GOOGLE_API_KEY'),
            temperature=0.2,
            max_retries=6
        )

        self.vector_store = None
        self.retrieval_chain = None

        self.prompt = ChatPromptTemplate.from_template(
            """You are a helpful medical assistant. Use the provided context to answer the user's question accurately.

            IMPORTANT: Provide the answer using clear bullet points and Markdown formatting. Be concise and direct.

            Context: {context}
            Question: {input}
            Answer:"""
        )

    def load_local_csv(self, file_path: str) -> List[Document]:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found at: {file_path}.")

        documents = []
        try:
            df = pd.read_csv(
                file_path,
                on_bad_lines='skip',
                engine='python',
                quoting=csv.QUOTE_MINIMAL,
                encoding='utf-8'
            )
            df.columns = [c.lower() for c in df.columns]
            df = df.dropna(subset=['question', 'answer'])

            for _, row in df.iterrows():
                content = f"Question: {row['question']}\nAnswer: {row['answer']}"
                documents.append(Document(page_content=content))

            print(f"Successfully loaded {len(documents)} medical documents.")
        except Exception as e:
            raise ValueError(f"Could not parse CSV file. Error: {e}")

        return documents

    def create_vector_store(self, documents: List[Document]):
        self.vector_store = FAISS.from_documents(documents, self.embeddings)

    def setup_retrieval_chain(self):
        document_chain = create_stuff_documents_chain(self.llm, self.prompt)
        self.retrieval_chain = create_retrieval_chain(
            self.vector_store.as_retriever(search_kwargs={"k": 5}),
            document_chain
        )

    def ask_question(self, question: str) -> str:
        if not self.retrieval_chain:
            raise ValueError("Chain not set up.")
        try:
            response = self.retrieval_chain.invoke({"input": question})
            return response["answer"]
        except Exception as e:
            if "429" in str(e):
                return "Error: Gemini API quota exceeded (429). Please wait a minute before trying again."
            return f"An error occurred: {e}"

print("MedQuadAssistant updated to provide point-wise answers.")

MedQuadAssistant updated to provide point-wise answers.


In [2]:
from sentence_transformers import CrossEncoder
import numpy as np
import re

class MedQuadAssistantWithReranker(MedQuadAssistant):
    def __init__(self):
        super().__init__()
        print("Loading Cross-Encoder reranker...")
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

        self.prompt = ChatPromptTemplate.from_template(
            """You are a strict professional medical assistant.

            SAFETY AND FAITHFULNESS RULES:
            - Use ONLY the provided context to answer.
            - Do NOT use outside medical knowledge.
            - If the context does not contain the answer, state: 'I am sorry, but the provided medical sources do not contain information to answer this specific question.'
            - Every bullet point MUST be followed by a citation like [Source X].

            INSTRUCTIONS:
            1. Provide the answer in bullet points.
            2. At the end, provide a Confidence Score: [CONFIDENCE: X.XX].

            Context:
            {context}

            Question: {input}
            Answer:"""
        )

    def calculate_retrieval_confidence(self, scores: List[float]) -> float:
        # Convert logit scores to probabilities using sigmoid
        probs = 1 / (1 + np.exp(-np.array(scores)))
        # Average the top probabilities
        return float(np.mean(probs[:3]))

    def ask_question(self, question: str) -> Dict[str, Any]:
        if not self.retrieval_chain:
            raise ValueError("Chain not set up.")

        try:
            retriever = self.vector_store.as_retriever(search_kwargs={"k": 10})
            initial_docs = retriever.invoke(question)

            model_inputs = [[question, doc.page_content] for doc in initial_docs]
            scores = self.reranker.predict(model_inputs)

            ranked_results = sorted(zip(scores, initial_docs), key=lambda x: x[0], reverse=True)
            top_scores = [score for score, _ in ranked_results]

            if len(top_scores) == 0 or top_scores[0] < -5:
                return {
                    "answer": "I am sorry, but I couldn't find any relevant medical documents.",
                    "confidence_score": 0.0,
                    "retrieval_quality": 0.0,
                    "sources_content": []
                }

            top_docs = [doc for _, doc in ranked_results][:5]
            retrieval_conf = self.calculate_retrieval_confidence(top_scores)

            context_with_sources = ""
            for i, doc in enumerate(top_docs):
                context_with_sources += f"--- Source {i+1} ---\n{doc.page_content}\n\n"

            response = self.llm.invoke(self.prompt.format(context=context_with_sources, input=question))
            raw_answer = response.content

            llm_conf = 0.5
            conf_match = re.search(r"\[CONFIDENCE: (\d\.\d+)\]", raw_answer)
            if conf_match:
                # Convert LLM 0-5 scale to 0-1 if needed, or handle as 0-1
                llm_val = float(conf_match.group(1))
                llm_conf = llm_val if llm_val <= 1.0 else llm_val / 5.0
                clean_answer = raw_answer.replace(conf_match.group(0), "").strip()
            else:
                clean_answer = raw_answer

            final_conf = (retrieval_conf * 0.4) + (min(llm_conf, 1.0) * 0.6)

            return {
                "answer": clean_answer,
                "confidence_score": round(final_conf * 100, 2),
                "retrieval_quality": round(retrieval_conf * 100, 2),
                "sources_content": [doc.page_content for doc in top_docs]
            }
        except Exception as e:
            return {"answer": f"Error: {e}", "confidence_score": 0, "sources_content": []}

print("MedQuadAssistant confidence calculation fixed.")

MedQuadAssistant confidence calculation fixed.


### Set OpenAI API Key

The `MedQuadAssistant` uses OpenAI models for embeddings and chat, so we need to ensure the `OPENAI_API_KEY` environment variable is correctly set.

In [3]:
# Ensure OPENAI_API_KEY is available in environment variables for OpenAI models
# The key was retrieved from Colab secrets as 'OPENAI_API_KEY' in the previous cell.
# Make sure to set the environment variable correctly for OpenAI's client.
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OPENAI_API_KEY environment variable set.")

OPENAI_API_KEY environment variable set.


###  Initializing main assistance `MedQuadAssistant` with re ranker and confidence estimation module



In [4]:
import csv
import pandas as pd
import numpy as np
from sentence_transformers import CrossEncoder
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata

try:
    print("🔄 Initializing main assistant with full 16,413 documents...")
    assistant = MedQuadAssistantWithReranker()

    # Robust loading to handle full dataset without truncation
    df = pd.read_csv(
        CSV_FILE_PATH,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines='skip',
        engine='python',
        encoding='utf-8-sig'
    ).dropna(subset=['question', 'answer'])

    docs = [Document(page_content=f"Question: {row['question']}\nAnswer: {row['answer']}") for _, row in df.iterrows()]

    # Initialize Vector Store with the full cleaned list
    assistant.create_vector_store(docs)
    assistant.retrieval_chain = True

    print(f"✅ Assistant successfully initialized with {len(docs)} documents!")
except Exception as e:
    print(f"❌ Initialization error: {e}")

🔄 Initializing main assistant with full 16,413 documents...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initializing Gemini 2.5 Flash...
Loading Cross-Encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Assistant successfully initialized with 16407 documents!


In [ ]:
try:
    import csv
    import pandas as pd

    print("🔄 Initializing assistant with all 16,413 documents...")
    assistant1 = MedQuadAssistantWithReranker()

    # Robust loading: use engine='python' and quotechar with doublequote=True to handle the medquad.csv format
    df = pd.read_csv(
        CSV_FILE_PATH,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines='skip',
        engine='python',
        encoding='utf-8-sig'
    ).dropna(subset=['question', 'answer'])

    # Prepare documents
    subset_docs = [Document(page_content=f"Question: {row['question']}\nAnswer: {row['answer']}") for _, row in df.iterrows()]

    # Limit to 16413 if the file contains more, or use what is available
    subset_docs = subset_docs[:16413]

    # Create vector store
    assistant1.create_vector_store(subset_docs)
    assistant1.retrieval_chain = True

    print(f"✅ assistant successfully initialized with {len(subset_docs)} documents!")

    # Verification test
    test_res = assistant1.ask_question("Briefly summarize what medical topics are covered in this dataset.")
    print(f"\nTest response: {test_res['answer']}")
    print("\nVerification check completed.")

except Exception as e:
    print(f"❌ Initialization error for assistant1: {e}")

🔄 Initializing assistant with all 16,413 documents...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ assistant successfully initialized with 16407 documents!

Test response: *   **Specific Conditions:**
    *   Chronic atrial and intestinal dysrhythmia [Source 1]
    *   Pulmonary alveolar microlithiasis [Source 2]
    *   Progressive familial heart block [Source 4]
    *   Hypochondrogenesis [Source 5]
    *   Interstitial Lung Disease [Source 2]
    *   Achondrogenesis [Source 5]
    *   Arrhythmia [Source 4]
    *   Sudden Cardiac Arrest [Source 4]

*   **General Medical Topics:**
    *   Diagnosis and management of various health conditions [Source 1, Source 2, Source 4, Source 5]
    *   Diagnostic Tests [Source 1, Source 2, Source 4, Source 5]
    *   Drug Therapy [Source 1, Source 2, Source 4, Source 5]
    *   Surgery and Rehabilitation [Source 1, Source 2, Source 4, Source 5]
    *   Genetic Counseling [Source 1, Source 2, Source 4, Source 5]
    *   Palliative Care [Source 1, Source 2, Source 4, Source 5]
    *   Pacemakers and Implantable Defibrillators [Source 1, Source 

In [ ]:
assistant = MedQuadAssistantWithReranker()

try:
    docs = assistant.load_local_csv(CSV_FILE_PATH)
    assistant.create_vector_store(docs)
    # Note: ask_question now handles retrieval manually for reranking flow
    assistant.retrieval_chain = True # Flag to indicate initialization is done
    print("Assistant with Reranker and Gemini 2.5 Flash is ready!")
except Exception as e:
    print(f"Error initializing: {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Assistant with Reranker and Gemini 2.5 Flash is ready!


In [ ]:
import sys
import pandas as pd
import csv

try:
    print("🔄 (Re)Initializing Assistant with Full Dataset and Citation Support...")
    assistant = MedQuadAssistantWithReranker()

    # Use the robust loading logic to ensure all 16,407 documents are loaded
    df = pd.read_csv(
        CSV_FILE_PATH,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines='skip',
        engine='python',
        encoding='utf-8-sig'
    ).dropna(subset=['question', 'answer'])

    docs = [Document(page_content=f"Question: {row['question']}\nAnswer: {row['answer']}") for _, row in df.iterrows()]

    assistant.create_vector_store(docs)
    assistant.retrieval_chain = True

    print(f"✅ Assistant is ready with {len(docs)} documents!")
    user_input = input("Enter your medical question: ")

    if user_input.lower() != 'exit' and user_input.strip():
        print(f"\n🔍 Generating answer with citations...")
        result = assistant.ask_question(user_input)

        print("\n" + "="*50)
        print("⚕️ ASSISTANT'S ANSWER")
        print("="*50)
        print(result['answer'])
        print("="*50)
        print(f"📊 Confidence: {result['confidence_score']}% | Retrieval Quality: {result['retrieval_quality']}%")
        print("="*50)

        # Optional: Show snippet of first source to verify retrieval
        if result['sources_content']:
             print(f"\nSource 1 Preview: {result['sources_content'][0][:200]}...")

except Exception as e:
    print(f"❌ Error: {e}")

🔄 (Re)Initializing Assistant with Full Dataset and Citation Support...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Assistant is ready with 16407 documents!


In [ ]:
# Final verification of the full RAG pipeline with Reranking
if 'assistant' in globals() and isinstance(assistant, MedQuadAssistantWithReranker):
    test_query = "What are the complications of untreated hypertension?"
    print(f"Testing Reranked Assistant with: {test_query}\n")
    try:
        answer = assistant.ask_question(test_query)
        print(f"--- Answer ---\n{answer}")
    except Exception as e:
        print(f"Test failed: {e}")
else:
    print("Assistant with Reranker not found. Please ensure all previous cells (installation and initialization) were executed successfully.")

Testing Reranked Assistant with: What are the complications of untreated hypertension?

--- Answer ---
*   Stroke
*   Heart failure
*   Heart attack
*   Kidney failure


In [ ]:
# Test the new Confidence Estimation Module
try:
    print("initializing assistant with Confidence Module...")
    assistant = MedQuadAssistantWithReranker()
    docs = assistant.load_local_csv(CSV_FILE_PATH)
    assistant.create_vector_store(docs)
    assistant.retrieval_chain = True

    test_query = "What are the main risk factors and symptoms of heart disease?"
    print(f"\n🔍 Querying: {test_query}")

    result = assistant.ask_question(test_query)

    print(f"\n--- Assistant's Answer ---\n{result['answer']}")
    print(f"\n--- Confidence Metrics ---")
    print(f"Overall Confidence Score: {result['confidence_score']}%")
    print(f"Retrieval Quality: {result['retrieval_quality']}%")

except Exception as e:
    print(f"❌ Error during test: {e}")

🔄 Re-initializing assistant with Confidence Module...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Initializing Gemini 2.5 Flash...
Loading Cross-Encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Successfully loaded 16407 medical documents.

🔍 Querying: What are the main risk factors and symptoms of heart disease?

--- Assistant's Answer ---
The main risk factors for heart disease include both factors you can control and those you cannot. The primary symptom mentioned is chest pain.

*   **Risk Factors You Can Control:**
    *   **Smoking:** Significantly increases the risk of heart disease [Source 1, Source 2, Source 3, Source 4, Source 5].
    *   **High Blood Pressure (Hypertension):** Strains the heart and damages blood vessels [Source 1, Source 2, Source 3, Source 4, Source 5].
    *   **High Blood Cholesterol and Abnormal Blood Fat Levels:** High LDL (bad) cholesterol and triglycerides, and low HDL (good) cholesterol, can lead to plaque buildup in arteries [Source 1, Source 2, Source 3, Source 4, Source 5].
    *   **Overweight and Obesity:** Especially central obesity (extra weight around the waist), increases the risk [Source 1, Source 2, Source 3, Source 4, Source 5].


### 🩺 Interactive Medical Assistant
Run the cell below to start chatting with the RAG system. It includes:
* **Reranking** for better accuracy.
* **Citation Generation** to show sources.
* **Confidence Estimation** to judge reliability.

In [ ]:
import time

def start_chat():
    if 'assistant' not in globals() or not hasattr(assistant, 'ask_question'):
        print("❌ Assistant not found. Please run the initialization cells first.")
        return

    print("--- Medical Assistant Ready (Type 'exit' to quit) ---")

    while True:
        query = input("\nPatient Query: ").strip()

        if query.lower() in ['exit', 'quit', 'stop']:
            print("Stopping chat...")
            break

        if not query:
            continue

        print(f"🔍 Analyzing sources and generating answer...")

        try:
            result = assistant.ask_question(query)

            print("\n" + "="*50)
            print("⚕️ ASSISTANT RESPONSE")
            print("="*50)
            print(result['answer'])
            print("="*50)
            print(f"📊 Confidence: {result['confidence_score']}% | Retrieval Quality: {result['retrieval_quality']}% ")
            print("="*50)

        except Exception as e:
            print(f"❌ An error occurred: {e}")

# Start the interaction
start_chat()

--- Medical Assistant Ready (Type 'exit' to quit) ---

Patient Query: what are the symptoms,treatment and prevention of malaria
🔍 Analyzing sources and generating answer...

⚕️ ASSISTANT RESPONSE
*   Malaria symptoms can range from absent or very mild to severe disease and death [Source 1].
*   Common symptoms include high fevers, shaking chills, and flu-like illness [Source 1].
*   Other symptoms may include chills, fever, vomiting, diarrhea, and jaundice [Source 2].
*   Malaria is a curable disease if diagnosed and treated promptly and correctly [Source 1].
*   Treatment involves drugs, and the specific type depends on factors such as disease severity, the species of malaria parasite, and where the infection was acquired [Source 1, Source 2].
*   Malaria can be prevented, especially when traveling to areas where it is found [Source 2].
*   Prevention methods include seeing a doctor for protective medicines, wearing insect repellent with DEET, covering up, and sleeping under mosquito 

In [5]:
import gradio as gr

def predict(message, history):
    """
    Chatbot-style function for multi-turn medical conversation.
    """
    if 'assistant' not in globals() or not hasattr(assistant, 'ask_question'):
        return "❌ Error: Assistant not initialized. Please run cell b90e43a7 first."

    if not message.strip():
        return "⚠️ Please enter a question."

    try:
        # Invoke the RAG assistant (Gemini 2.5 Flash + Reranker)
        result = assistant.ask_question(message)

        ans = result.get('answer', 'No answer generated.')
        conf = result.get('confidence_score', 'N/A')
        ret_q = result.get('retrieval_quality', 'N/A')

        # Format response with Markdown and collapsible sources
        response_md = f"{ans}\n\n"
        response_md += "---\n"
        response_md += f"📊 **Confidence:** {conf}% | **Retrieval Quality:** {ret_q}%\n\n"

        if 'sources_content' in result and result['sources_content']:
            response_md += "**Sources:**\n"
            for i, text in enumerate(result['sources_content']):
                # Keep payload light (800 chars) to prevent tunnel/parse errors
                display_text = text[:800] + "..." if len(text) > 800 else text
                response_md += f"<details><summary>Source {i+1}</summary> {display_text}</details>"

        return response_md
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Setup ChatInterface
demo = gr.ChatInterface(
    fn=predict,
    title="⚕️ MedQuad Multi-Turn Assistant",
    description="Ask follow-up medical questions. Sources and confidence scores are provided for transparency.",
    examples=["What are the symptoms of asthma?", "How is diabetes treated?", "How to prevent malaria?"],
    cache_examples=False
)

# Relaunching with Soft theme applied via launch
demo.launch(share=True, debug=True, inline=True, theme=gr.themes.Soft())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4febb81a8498017c12.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4febb81a8498017c12.gradio.live


### Verify LangChain Installation


In [ ]:
import langchain
from langchain.chains import create_retrieval_chain

print(f"LangChain version: {langchain.__version__}")
print("Successfully imported 'langchain.chains' and its sub-modules.")
print("LangChain installation verified!")

LangChain version: 0.1.20
Successfully imported 'langchain.chains' and its sub-modules.
LangChain installation verified!


In [ ]:
import langchain
print(f"Langchain version: {langchain.__version__}")

Langchain version: 0.1.20


In [8]:
# Perform a clean uninstall of potentially conflicting packages
!pip uninstall -y langchain ragas langchain-google-vertexai langchain-community langchain-core langchain-text-splitters langsmith openai langchain-google-genai

# Install all specific versions at once in quiet mode (-q) to prevent UI/Pathlib parsing errors in Colab
!pip install -q \
    "openai<2.0.0" \
    "langchain==0.1.20" \
    "langchain-core==0.1.53" \
    "langchain-community==0.0.38" \
    "langchain-text-splitters==0.0.2" \
    "langsmith==0.1.147" \
    "ragas==0.4.3" \
    "langchain-google-genai" \
    chromadb pypdf datasets GitPython faiss-cpu sentence-transformers

print("\n✅ All packages installed successfully. Please restart the Colab runtime (Runtime -> Restart runtime) before proceeding.")

Found existing installation: langchain 0.1.20
Uninstalling langchain-0.1.20:
  Successfully uninstalled langchain-0.1.20
Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
Found existing installation: langchain-community 0.0.38
Uninstalling langchain-community-0.0.38:
  Successfully uninstalled langchain-community-0.0.38
Found existing installation: langchain-core 0.1.53
Uninstalling langchain-core-0.1.53:
  Successfully uninstalled langchain-core-0.1.53
Found existing installation: langchain-text-splitters 0.0.2
Uninstalling langchain-text-splitters-0.0.2:
  Successfully uninstalled langchain-text-splitters-0.0.2
Found existing installation: langsmith 0.1.147
Uninstalling langsmith-0.1.147:
  Successfully uninstalled langsmith-0.1.147
Found existing installation: openai 1.109.1
Uninstalling openai-1.109.1:
  Successfully uninstalled openai-1.109.1
Found existing installation: langchain-google-genai 1.0.4
Uninstalling langchain-goo